# Processamento Batch vs. Streaming

## Preparação - tabelas de trabalho e checkpoint

O conjunto de dados `samples.wanderbricks.clickstream` contém dados simulados de navegação de usuários em uma plataforma de reservas de hospedagens.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Cria os caminhos do catálogo e esquema
catalogo = "senai"
esquema = "batch_streaming"
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")

origem_samples = "samples.wanderbricks.clickstream"
amostra = f"{catalogo}.{esquema}.amostra"
fonte = f"{catalogo}.{esquema}.fonte"
destino = f"{catalogo}.{esquema}.destino"
volume = "streaming"

# Checkpoint da atividade
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.{esquema}.{volume}")
checkpoint = f"/Volumes/{catalogo}/{esquema}/{volume}/checkpoint_clickstream"

# Reinicia as tabelas e volume
dbutils.fs.rm(checkpoint, True)
spark.sql(f"DROP TABLE IF EXISTS {destino}")
spark.sql(f"DROP TABLE IF EXISTS {fonte}")
spark.sql(f"DROP TABLE IF EXISTS {amostra}")

# Seleciona registros da tabela pública. O id_evento é criado para dividir os lotes
base = (spark.table(origem_samples)
    .select(F.col("event").alias("evento"),
            F.col("metadata.device").alias("dispositivo"))
    .where(F.col("evento").isNotNull())
    .limit(120)
    .withColumn("id_evento", F.row_number().over(Window.orderBy(F.rand(seed=22))))
    .filter(F.col("id_evento") <= 30)
    .select("id_evento", "evento", "dispositivo"))

base.write.format("delta").mode("overwrite").saveAsTable(amostra)

# Tabela Delta vazia: receberá os eventos à medida que liberarmos cada lote
spark.table(amostra).limit(0).write.format("delta").mode("overwrite").saveAsTable(fonte)

print("Fonte pública:", origem_samples)
print("Tabela de entrada:", fonte)
print("Tabela de saída:", destino)
print("Eventos selecionados:", spark.table(amostra).count())
display(spark.table(amostra).orderBy("id_evento"))

## Ciclo 1 - Leitura batch

Vamos liberar os **12 primeiros eventos** na tabela de entrada e fazer uma leitura batch. O resultado calculado nesta etapa não muda sozinho quando novos registros chegarem.

In [0]:
# Primeira chegada: 12 eventos do dataset público
lote1 = spark.table(amostra).filter(F.col("id_evento").between(1, 12))
lote1.write.format("delta").mode("append").saveAsTable(fonte)

# Processamento batch inicial
#TODO


total_inicial = df_batch_inicial.count()
print("Quantidade de eventos na primeira análise batch:", total_inicial)
display(df_batch_inicial.orderBy("id_evento"))

**Atividade 1 - Eventos por tipo**

Usando `df_batch_inicial`, calcule a quantidade de eventos agrupados por `evento`, em ordem decrescente de frequência. Qual evento aparece mais vezes neste primeiro conjunto?

In [0]:
# Atividade 1


## Ciclo 2 - Novos eventos e nova leitura batch

Vamos acrescentar mais **8 eventos** à mesma tabela de entrada.

In [0]:
# Segunda chegada: mais 8 eventos (IDs 13 a 20).
lote2 = spark.table(amostra).filter(F.col("id_evento").between(13, 20))
lote2.write.format("delta").mode("append").saveAsTable(fonte)

# Processamento batch atualizado
#TODO


print("Resultado da primeira análise batch (já calculado):", total_inicial)
print("Nova leitura batch:", df_batch_atualizado.count())
display(df_batch_atualizado.orderBy("id_evento"))

**Atividade 2 - Atualização dos indicadores**

Na nova leitura, conte os eventos por `dispositivo`.

In [0]:
# Atividade 2


## Ciclo 3 - Structured Streaming em tabela Delta

Ao utilizar o `readStream`, o processamento acompanha uma origem que admite novos registros. O checkpoint registra o progresso da consulta. A primeira execução processa os **20 eventos já disponíveis**.

In [0]:
# Lê incrementalmente a tabela Delta de entrada
df_stream = spark.readStream.table(fonte)

#TODO


print("Eventos persistidos pelo streaming:", spark.table(destino).count())
display(spark.table(destino).orderBy("id_evento"))

**Atividade 3 - Consultando a saída do streaming**

Consulte a tabela `destino` com `spark.table()` e conte os eventos por `evento`.

In [0]:
# Atividade 3


## Ciclo 4 - Retomada com o mesmo checkpoint

A próxima execução do streaming deve consumir somente os eventos acrescentados desde o último processamento.

In [0]:
# Terceira chegada: mais 10 eventos (IDs 21 a 30).
lote3 = spark.table(amostra).filter(F.col("id_evento").between(21, 30))

lote3.write.format("delta").mode("append").saveAsTable(fonte)

print("Novos eventos adicionados à fonte:", lote3.count())
print("Total da fonte:", spark.table(fonte).count())

In [0]:
# Quantidade de eventos antes do processamento
antes = spark.table(destino).count()

# Retoma o streaming utilizando o mesmo checkpoint
#TODO


# Verificação
print("Eventos anteriores:", antes)
print("Eventos atuais:", spark.table(destino).count())

## Ciclo 5 - Ingestão de arquivos com Auto Loader

Agora utilizaremos o **Auto Loader**, que identifica e processa novos **arquivos** em uma pasta. A demonstração grava **10 eventos em JSON** e utiliza `cloudFiles` para carregá-los em outra tabela Delta. O checkpoint permite continuar a ingestão quando chegarem novos arquivos.

In [0]:
# Preparação da pasta de arquivos e da tabela de saída
pasta_json = f"/Volumes/{catalogo}/{esquema}/{volume}/json"
checkpoint_autoloader = f"/Volumes/{catalogo}/{esquema}/{volume}/checkpoint_autoloader"
tabela_autoloader = f"{catalogo}.{esquema}.autoloader"

# Reinicia somente os objetos deste ciclo
dbutils.fs.rm(checkpoint_autoloader, True)
dbutils.fs.rm(pasta_json, True)
spark.sql(f"DROP TABLE IF EXISTS {tabela_autoloader}")

# Disponibiliza os primeiros 10 eventos como arquivos JSON
(
    spark.table(amostra)
        .filter(F.col("id_evento")
        .between(1, 10))
        .write.mode("overwrite")
        .json(pasta_json)
)

# Auto Loader: descobre arquivos JSON e grava os registros em Delta
#TODO


# Confere os resultados
print("Eventos ingeridos por Auto Loader:", spark.table(tabela_autoloader).count())
display(spark.table(tabela_autoloader).orderBy("id_evento"))

**Atividade 4 - Novos arquivos JSON**

Grave os eventos de `id_evento` **11 a 15** na mesma pasta `pasta_json`, usando a escrita em modo `append`. Em seguida, execute novamente o Auto Loader com **o mesmo checkpoint** e confirme que a tabela `tabela_autoloader` contém **15 eventos**, sem duplicar os 10 anteriores.

In [0]:
# Novos arquivos no mesmo checkpoint
(spark.table(amostra)
    .filter(F.col("id_evento").between(11, 15))
    .write.mode("append").json(pasta_json))

In [0]:
# Atividade 5


print("Total após a nova chegada:", spark.table(tabela_autoloader).count())
display(spark.table(tabela_autoloader).orderBy("id_evento"))